# 🔍 Debug — Phase 1: Adaptive RAG Agent

Notebook kiểm tra toàn bộ pipeline:
1. **Config** — settings & imports
2. **QueryClassifier** — strategy routing
3. **search_by_metadata()** — CustomSearch mới
4. **AdaptiveRAGAgent** — 3 strategies
5. **Orchestrator._get_rag_context()** — integration

In [1]:
import sys, os
from pathlib import Path

# Setup project root
ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from dotenv import load_dotenv
load_dotenv(ROOT / '.env')

print(f'✅ Project root: {ROOT}')
print(f'✅ .env loaded')

✅ Project root: c:\Users\Admin\OneDrive - Hanoi University of Science and Technology\Desktop\ĐATN
✅ .env loaded


---
## 1. Config & Imports

In [2]:
from src.config.config import settings

print('=== Settings ===')
print(f'  RETRIEVER_TOP_K      = {settings.RETRIEVER_TOP_K}')
print(f'  RERANKER_TOP_N       = {settings.RERANKER_TOP_N}')
print(f'  RAG_BROAD_MAX_CHUNKS = {settings.RAG_BROAD_MAX_CHUNKS}')
print(f'  LLM_MODEL            = {settings.LLM_MODEL}')
print(f'  GENAI_API_KEY set    = {bool(settings.GENAI_API_KEY)}')

=== Settings ===
  RETRIEVER_TOP_K      = 25
  RERANKER_TOP_N       = 5
  RAG_BROAD_MAX_CHUNKS = 30
  LLM_MODEL            = gemini-2.5-flash-lite
  GENAI_API_KEY set    = True


In [3]:
from src.rag.adaptive_rag import AdaptiveRAGAgent, RAGStrategy, RAGResult, QueryClassifier
from src.rag.retrieve_rebuild import CustomSearch
from src.rag.reranker import Reranker
from src.llm.orchestrator import Orchestrator

print('✅ All imports successful')

✅ All imports successful


---
## 2. QueryClassifier — Strategy Routing

Test xem classifier có chọn đúng strategy không.

In [4]:
clf = QueryClassifier()

test_cases = [
    # (query, intent_hint, expected_strategy, expected_grade)
    ('mạng LAN là gì',               'explain',  RAGStrategy.STANDARD,   None),
    ('tổng hợp kiến thức lớp 12',    'explain',  RAGStrategy.BROAD,      '12'),
    ('lớp 12 có những gì',           'explain',  RAGStrategy.BROAD,      '12'),
    ('tổng quan lớp 10',             'explain',  RAGStrategy.BROAD,      '10'),
    ('tạo 3 câu mcq về mạng máy tính','generate', RAGStrategy.STANDARD,  None),
    ('xin chào',                     'chat',     RAGStrategy.STANDARD,   None),
    ('nội dung chương trình lớp 11', 'explain',  RAGStrategy.CURRICULUM, '11'),
    ('danh sách bài lớp 10',         'explain',  RAGStrategy.CURRICULUM, '10'),
]

print(f'{"Query":<45} {"Intent":<10} {"Expected":<12} {"Got":<12} {"Grade":<8} {"OK"}')
print('-' * 100)

all_ok = True
for query, intent, exp_strat, exp_grade in test_cases:
    profile = clf.classify(query, intent_hint=intent)
    strat_ok  = profile.strategy == exp_strat
    grade_ok  = profile.grade == exp_grade
    ok = strat_ok and grade_ok
    if not ok:
        all_ok = False
    icon = '✅' if ok else '❌'
    print(f'{query[:44]:<45} {intent:<10} {exp_strat.value:<12} {profile.strategy.value:<12} {str(profile.grade):<8} {icon}')
    if not ok:
        print(f'   → reason: {profile.reason}')

print()
print('✅ All passed!' if all_ok else '❌ Some tests FAILED — check rows above')

Query                                         Intent     Expected     Got          Grade    OK
----------------------------------------------------------------------------------------------------
mạng LAN là gì                                explain    standard     standard     None     ✅
tổng hợp kiến thức lớp 12                     explain    broad        broad        12       ✅
lớp 12 có những gì                            explain    broad        broad        12       ✅
tổng quan lớp 10                              explain    broad        broad        10       ✅
tạo 3 câu mcq về mạng máy tính                generate   standard     standard     None     ✅
xin chào                                      chat       standard     standard     None     ✅
nội dung chương trình lớp 11                  explain    curriculum   curriculum   11       ✅
danh sách bài lớp 10                          explain    curriculum   curriculum   10       ✅

✅ All passed!


---
## 3. Load CustomSearch

> ⚠️ Cell này mất ~10-20s để load embeddings.

In [5]:
import time

CHUNKS_PATH = str(ROOT / 'data' / 'rag_chunks_v2.json')
EMBED_PATH  = str(ROOT / 'data' / 'embeddings.npy')

t0 = time.time()
searcher = CustomSearch(chunks_path=CHUNKS_PATH, embeddings_path=EMBED_PATH)
print(f'✅ Loaded {searcher.corpus_size} chunks in {time.time()-t0:.2f}s')

CustomSearch initialized: 2348 docs, vocab=17139, avgdl=140.9
✅ Loaded 2348 chunks in 0.68s


---
## 4. Test `search_by_metadata()`

So sánh speed và kết quả với standard search.

In [6]:
from collections import Counter

# --- Test 1: Grade filter ---
t0 = time.time()
r12 = searcher.search_by_metadata(grade='12')
t_meta = time.time() - t0

grades_found = Counter(c['metadata'].get('grade') for c in r12)
print(f'=== grade="12" filter ===')
print(f'  Chunks returned : {len(r12)}')
print(f'  Time            : {t_meta*1000:.1f}ms')
print(f'  Grades found    : {dict(grades_found)}')
print(f'  All grade=12    : {all(c["metadata"].get("grade")=="12" for c in r12)}')

# Sample output
print(f'\n  Sample chunk:')
if r12:
    s = r12[0]
    print(f'    topic   : {s["metadata"].get("topic_name","")[:60]}')
    print(f'    lesson  : {s["metadata"].get("lesson_name","")[:60]}')
    print(f'    type    : {s["metadata"].get("type","")}')
    print(f'    content : {s["content"][:120]}...')

=== grade="12" filter ===
  Chunks returned : 126
  Time            : 2.0ms
  Grades found    : {'12': 126}
  All grade=12    : True

  Sample chunk:
    topic   : Máy tính và xã hội tri thức – Giới thiệu trí tuệ nhân tạo
    lesson  : 
    type    : objective
    content : Học xong bài này, em sẽ:
* Giải thích được sơ lược về khái niệm Trí tuệ nhân tạo (AI).
* Nêu được ví dụ để thấy một hệ t...


In [7]:
# --- Test 2: grade + objective chunks ---
t0 = time.time()
r_obj = searcher.search_by_metadata(grade='12', chunk_types=['objective'], max_per_lesson=1)
t_obj = time.time() - t0

types_found = Counter(c['metadata'].get('type') for c in r_obj)
lessons = [c['metadata'].get('lesson_name','') for c in r_obj]
topics  = list(dict.fromkeys(c['metadata'].get('topic_name','') for c in r_obj))  # unique ordered

print(f'=== grade="12" + objective + max_per_lesson=1 ===')
print(f'  Chunks returned : {len(r_obj)}')
print(f'  Time            : {t_obj*1000:.1f}ms')
print(f'  Types found     : {dict(types_found)}')
print(f'  All objective   : {all(c["metadata"].get("type")=="objective" for c in r_obj)}')
print(f'  Unique lessons  : {len(set(lessons))}')
print(f'  Unique topics   : {len(set(topics))}')

print(f'\n  Topics (grade 12):')
for t in topics:
    print(f'    • {t[:80]}')

=== grade="12" + objective + max_per_lesson=1 ===
  Chunks returned : 56
  Time            : 1.0ms
  Types found     : {'objective': 56}
  All objective   : True
  Unique lessons  : 56
  Unique topics   : 13

  Topics (grade 12):
    • Máy tính và xã hội tri thức – Giới thiệu trí tuệ nhân tạo
    • Mạng máy tính và Internet – Kết nối mạng
    • Giải quyết vấn đề với sự trợ giúp của máy tính – Tạo trang Web
    • Hướng nghiệp với Tin học – Giới thiệu nhóm nghề Dịch vụ và Quản trị
    • Mạng máy tính và Internet – Phác thảo thiết kế mạng máy tính
    • Giải quyết vấn đề với sự trợ giúp của máy tính – Giới thiệu học máy và khoa học 
    • Giải quyết vấn đề với sự trợ giúp của máy tính – Giới thiệu học máy và khoa học 
    • Máy tính và xã hội tri thức
    • Mạng máy tính và Internet
    • Đạo đức, pháp luật và văn hoá trong môi trường số
    • Giải quyết vấn đề với sự trợ giúp của máy tính
    • Hướng nghiệp với Tin học
    • Ứng dụng Tin học


In [8]:
# --- Test 3: max_per_lesson enforcement ---
r_limited = searcher.search_by_metadata(grade='12', max_per_lesson=1)
lesson_count = Counter(c['metadata'].get('lesson_name','') for c in r_limited)
max_per = max(lesson_count.values()) if lesson_count else 0

print(f'=== max_per_lesson=1 test ===')
print(f'  Max count per lesson : {max_per}  (must be ≤ 1)')
print(f'  ✅ OK' if max_per <= 1 else f'  ❌ FAIL — got {max_per}')

# Speed comparison
print(f'\n=== Speed vs standard search ===')
t0 = time.time()
r_standard = searcher.search('tổng hợp kiến thức lớp 12', top_k=25)
t_standard = time.time() - t0

t0 = time.time()
r_meta = searcher.search_by_metadata(grade='12', chunk_types=['objective'], max_per_lesson=1)
t_meta = time.time() - t0

print(f'  standard search  : {len(r_standard)} chunks, {t_standard:.2f}s')
print(f'  metadata filter  : {len(r_meta)} chunks, {t_meta*1000:.1f}ms')
print(f'  Speedup          : {t_standard/t_meta:.0f}x faster' if t_meta > 0 else 'N/A')

=== max_per_lesson=1 test ===
  Max count per lesson : 1  (must be ≤ 1)
  ✅ OK

=== Speed vs standard search ===
Loading embedding model: dangvantuan/vietnamese-document-embedding...
Model loaded on cuda
  standard search  : 25 chunks, 14.15s
  metadata filter  : 56 chunks, 0.0ms
N/A


---
## 5. AdaptiveRAGAgent — 3 Strategies

> ⚠️ Load reranker lần đầu sẽ mất ~30-60s.

In [9]:
print('Loading Reranker...')
t0 = time.time()
reranker = Reranker()
print(f'✅ Reranker loaded in {time.time()-t0:.2f}s')

agent = AdaptiveRAGAgent(
    retriever=searcher,
    reranker=reranker,
    settings=settings,
)
print('✅ AdaptiveRAGAgent ready')

Loading Reranker...
✅ Reranker loaded in 0.00s
✅ AdaptiveRAGAgent ready


In [10]:
def show_rag_result(result, label=''):
    print(f'\n{"="*60}')
    print(f'  {label}')
    print(f'{"="*60}')
    print(f'  Strategy  : {result.strategy_used.value}')
    print(f'  Chunks    : {len(result.chunks)}')
    print(f'  Time      : {result.total_time_s}s')
    print(f'  Filter    : {result.metadata_filter}')
    print(f'  Reason    : {result.reason}')
    if result.chunks:
        print(f'\n  Top 3 chunks:')
        for i, c in enumerate(result.chunks[:3]):
            meta = c.get('metadata', {})
            print(f'  [{i+1}] {meta.get("lesson_name","N/A")[:50]}')
            print(f'       {c["content"][:120].strip()}...')

print('Helper function ready')

Helper function ready


In [11]:
# STANDARD: query cụ thể → BM25+Semantic+Rerank
t0 = time.time()
r_std = agent.retrieve('mạng LAN là gì', intent_hint='explain')
t_std = time.time() - t0

show_rag_result(r_std, f'STANDARD — "mạng LAN là gì" ({t_std:.2f}s)')

[23:45:10] INFO    | RAGAgent: strategy=standard | Query cụ thể


Loading reranker: AITeamVN/Vietnamese_Reranker...
Reranker loaded on cuda


[23:45:33] INFO    | RAGAgent done: 5 chunks, 22.88s



  STANDARD — "mạng LAN là gì" (22.88s)
  Strategy  : standard
  Chunks    : 5
  Time      : 22.88s
  Filter    : {'grade': None, 'topic': None}
  Reason    : Query cụ thể

  Top 3 chunks:
  [1] CƠ SỞ VỀ MẠNG MÁY TÍNH
       Mạng LAN (Local Area Network) hay còn gọi là **mạng cục bộ** là loại mạng kết nối những máy tính và các thiết bị số tron...
  [2] ĐIỆN TOÁN ĐÁM MÂY VÀ INTERNET VẠN VẬT
       Em đã biết mạng **LAN** (Local Area Network – mạng cục bộ) là mạng của một cơ quan hay gia đình, chỉ kết nối những máy t...
  [3] CƠ SỞ VỀ MẠNG MÁY TÍNH
       Mạng WLAN (Wireless Local Area Network) hay còn gọi là **mạng cục bộ không dây** là một loại mạng cục bộ sử dụng công ng...


In [12]:
# BROAD: query tổng quát → metadata filter (cực nhanh)
t0 = time.time()
r_broad = agent.retrieve('tổng hợp kiến thức lớp 12', intent_hint='explain')
t_broad = time.time() - t0

show_rag_result(r_broad, f'BROAD — "tổng hợp kiến thức lớp 12" ({t_broad:.2f}s)')

print(f'\n  ⚡ Speedup vs STANDARD: {t_std/t_broad:.0f}x faster' if t_broad > 0 else '')

[23:45:33] INFO    | RAGAgent: strategy=broad | Query tổng quát: broad=True, grade_only=True
[23:45:33] INFO    | RAGAgent done: 30 chunks, 0.01s



  BROAD — "tổng hợp kiến thức lớp 12" (0.01s)
  Strategy  : broad
  Chunks    : 30
  Time      : 0.01s
  Filter    : {'grade': '12', 'topic': None}
  Reason    : Query tổng quát: broad=True, grade_only=True

  Top 3 chunks:
  [1] 
       Học xong bài này, em sẽ:
* Giải thích được sơ lược về khái niệm Trí tuệ nhân tạo (AI).
* Nêu được ví dụ để thấy một hệ t...
  [2] GIỚI THIỆU VỀ TRÍ TUỆ NHÂN TẠO
       *   Chỉ ra được một số lĩnh vực của khoa học công nghệ, đời sống đã và đang phát triển mạnh mẽ dựa trên những thành tựu...
  [3] CƠ SỞ VỀ MẠNG MÁY TÍNH
       Học xong bài này, em sẽ:
* Nêu được chức năng chính của một số thiết bị mạng thông dụng: Access Point, Switch, Modem, Ro...

  ⚡ Speedup vs STANDARD: 2540x faster


In [13]:
# CURRICULUM: hỏi cấu trúc chương trình → synthetic chunks
t0 = time.time()
r_cur = agent.retrieve('nội dung chương trình lớp 10', intent_hint='explain')
t_cur = time.time() - t0

show_rag_result(r_cur, f'CURRICULUM — "nội dung chương trình lớp 10" ({t_cur:.2f}s)')

[23:45:33] INFO    | RAGAgent: strategy=curriculum | Hỏi về cấu trúc chương trình
[23:45:33] INFO    | RAGAgent done: 25 chunks, 0.01s



  CURRICULUM — "nội dung chương trình lớp 10" (0.01s)
  Strategy  : curriculum
  Chunks    : 25
  Time      : 0.01s
  Filter    : {'grade': '10', 'topic': None}
  Reason    : Hỏi về cấu trúc chương trình

  Top 3 chunks:
  [1] DỮ LIỆU, THÔNG TIN VÀ XỬ LÍ THÔNG TIN
       Chủ đề: Máy tính và xã hội tri thức – Tin học và xử lí thông tin
Bài học: DỮ LIỆU, THÔNG TIN VÀ XỬ LÍ THÔNG TIN
---
Học...
  [2] SỰ ƯU VIỆT CỦA MÁY TÍNH VÀ NHỮNG THÀNH TỰU CỦA TIN
       Chủ đề: Máy tính và xã hội tri thức – Tin học và xử lí thông tin
Bài học: SỰ ƯU VIỆT CỦA MÁY TÍNH VÀ NHỮNG THÀNH TỰU CỦA...
  [3] THỰC HÀNH SỬ DỤNG THIẾT BỊ SỐ
       Chủ đề: Máy tính và xã hội tri thức – Tin học và xử lí thông tin
Bài học: THỰC HÀNH SỬ DỤNG THIẾT BỊ SỐ
---
*   Biết đượ...


In [14]:
# So sánh tổng hợp
print('=== STRATEGY COMPARISON ===')
print(f'{"Strategy":<12} {"Chunks":<8} {"Time":<12} {"Use case"}')
print('-' * 60)
print(f'{"STANDARD":<12} {len(r_std.chunks):<8} {t_std:.2f}s        Query cụ thể (BM25+Semantic+Rerank)')
print(f'{"BROAD":<12} {len(r_broad.chunks):<8} {t_broad:.3f}s       Query tổng quát (metadata filter)')
print(f'{"CURRICULUM":<12} {len(r_cur.chunks):<8} {t_cur:.3f}s       Cấu trúc chương trình')

=== STRATEGY COMPARISON ===
Strategy     Chunks   Time         Use case
------------------------------------------------------------
STANDARD     5        22.88s        Query cụ thể (BM25+Semantic+Rerank)
BROAD        30       0.009s       Query tổng quát (metadata filter)
CURRICULUM   25       0.008s       Cấu trúc chương trình


---
## 6. Orchestrator Integration

Kiểm tra `_get_rag_context()` và `last_debug_info` JSON trace.

In [15]:
import json

orch = Orchestrator(retriever=searcher, reranker=reranker)

print('Orchestrator attributes:')
print(f'  has rag_agent          : {hasattr(orch, "rag_agent")}')
print(f'  rag_agent type         : {type(orch.rag_agent).__name__}')
print(f'  rag_agent.classifier   : {type(orch.rag_agent.classifier).__name__}')

Orchestrator attributes:
  has rag_agent          : True
  rag_agent type         : AdaptiveRAGAgent
  rag_agent.classifier   : QueryClassifier


In [16]:
# Test _get_rag_context với 3 intent khác nhau
test_queries = [
    ('mạng LAN là gì',              'explain'),
    ('tổng hợp kiến thức lớp 12',   'explain'),
    ('tạo câu hỏi về mạng máy tính', 'generate'),
]

for query, intent in test_queries:
    orch.last_debug_info = {'steps': []}  # reset trace
    chunks = orch._get_rag_context(query, intent_hint=intent)
    
    rag_step = next((s for s in orch.last_debug_info['steps'] if s.get('node') == 'RAG'), {})
    
    print(f'Query  : "{query[:50]}"')
    print(f'Intent : {intent}')
    print(f'Chunks : {len(chunks)}')
    print(f'Trace  : {json.dumps(rag_step, ensure_ascii=False)}')
    print()

[23:45:56] INFO    | RAGAgent: strategy=standard | Query cụ thể
[23:46:05] INFO    | RAGAgent done: 5 chunks, 9.01s
[23:46:05] INFO    | RAGAgent: strategy=broad | Query tổng quát: broad=True, grade_only=True
[23:46:05] INFO    | RAGAgent done: 30 chunks, 0.00s
[23:46:05] INFO    | RAGAgent: strategy=standard | Query cụ thể


Query  : "mạng LAN là gì"
Intent : explain
Chunks : 5
Trace  : {"node": "RAG", "strategy": "standard", "chunks_returned": 5, "time_s": 9.01, "filter": {"grade": null, "topic": null}, "reason": "Query cụ thể"}

Query  : "tổng hợp kiến thức lớp 12"
Intent : explain
Chunks : 30
Trace  : {"node": "RAG", "strategy": "broad", "chunks_returned": 30, "time_s": 0.0, "filter": {"grade": "12", "topic": null}, "reason": "Query tổng quát: broad=True, grade_only=True"}



[23:46:13] INFO    | RAGAgent done: 5 chunks, 7.73s


Query  : "tạo câu hỏi về mạng máy tính"
Intent : generate
Chunks : 5
Trace  : {"node": "RAG", "strategy": "standard", "chunks_returned": 5, "time_s": 7.73, "filter": {"grade": null, "topic": null}, "reason": "Query cụ thể"}



---
## 7. End-to-End: Full Pipeline Test

Chạy `ask()` hoàn chỉnh và xem `pipeline_trace.log`.

In [17]:
from src.llm.memory import MemoryManager

# Reset memory
orch.memory = MemoryManager()

QUERY = 'tổng hợp kiến thức tin học lớp 12'

print('='*70)
print(f'QUERY: "{QUERY}"')
print('='*70)

t0 = time.time()
chunks_all = []
for chunk in orch.ask(QUERY):
    chunks_all.append(chunk)

elapsed = time.time() - t0
full_response = ''.join(chunks_all)

print(f'\n{'='*70}')
print(f'RESPONSE ({len(full_response)} chars, {elapsed:.2f}s):')
print('='*70)
preview = full_response[:2000]
print(preview)
if len(full_response) > 2000:
    print(f'\n... [{len(full_response)-2000} chars truncated]')

[23:46:13] INFO    | ============================================================
[23:46:13] INFO    | QUERY: 'tổng hợp kiến thức tin học lớp 12'


QUERY: "tổng hợp kiến thức tin học lớp 12"


[23:46:14] INFO    | IntentRouter: intent=explain, task_type=None, topic=kiến thức tin học lớp 12, is_new_topic=True
[23:46:14] INFO    | IntentRouter (1.10s): intent=explain, task_type=None, topic=kiến thức tin học lớp 12, is_new_topic=True
[23:46:14] INFO    | No current session, creating new
[23:46:14] INFO    | New session created: id=5fe0b7c0, topic='kiến thức tin học lớp 12', intent=explain
[23:46:14] INFO    | Session: id=5fe0b7c0, topic='kiến thức tin học lớp 12', msgs=0
[23:46:14] INFO    | ActionPlan: explain_concept (General concept explanation)
[23:46:14] INFO    | RAGAgent: strategy=broad | Query tổng quát: broad=True, grade_only=True
[23:46:14] INFO    | RAGAgent done: 30 chunks, 0.00s
[23:46:25] INFO    | Total time: 11.88s
[23:46:25] INFO    | ============================================================



RESPONSE (16514 chars, 11.88s):
Dang tim tai lieu de giai thich...

Chào em, thầy là EduBot, trợ lý học tập Tin học THPT của em đây! Thầy rất vui được đồng hành cùng em ôn tập kiến thức Tin học lớp 12. Hôm nay, chúng ta sẽ cùng nhau hệ thống lại những kiến thức quan trọng nhé!

Để giúp em dễ dàng theo dõi, thầy sẽ tổng hợp kiến thức theo từng chủ đề lớn, mỗi chủ đề sẽ được giải thích theo cấu trúc: **Khái niệm cốt lõi**, **Giải thích chi tiết**, **Ví dụ minh họa**, và **Tóm tắt**.

---

### Chủ đề 1: Trí tuệ nhân tạo (AI)

**1. Khái niệm cốt lõi:**
Trí tuệ nhân tạo (AI) là lĩnh vực khoa học máy tính tập trung vào việc tạo ra các hệ thống có khả năng thực hiện các nhiệm vụ mà thông thường đòi hỏi trí tuệ con người, như học hỏi, suy luận, giải quyết vấn đề, nhận thức và hiểu ngôn ngữ.

**2. Giải thích chi tiết:**
AI không phải là một khái niệm duy nhất, mà là một tập hợp các công nghệ và phương pháp khác nhau. Một hệ thống AI thường có các khả năng sau:
*   **Tri thức:** Hệ thống có khả

In [18]:
# Xem pipeline_trace.log
trace_file = ROOT / 'logs' / 'pipeline_trace.log'

if trace_file.exists():
    with open(trace_file, 'r', encoding='utf-8') as f:
        trace = json.load(f)
    
    print(f'=== PIPELINE TRACE ===')
    print(f'Query     : {trace.get("query","").strip()}')
    print(f'Timestamp : {trace.get("timestamp")}')
    print(f'Total time: {trace.get("total_time_s")}s')
    
    print(f'\nSteps ({len(trace.get("steps",[]))}):')
    for i, step in enumerate(trace.get('steps', [])):
        node = step.get('node', '?')
        # Format key info per node type
        if node == 'RAG':
            info = f'strategy={step.get("strategy")}, chunks={step.get("chunks_returned")}, time={step.get("time_s")}s'
        elif node == 'IntentRouter':
            info = f'intent={step.get("primary_intent")}, topic={step.get("topic")}, time={step.get("time_s")}s'
        elif node == 'ActionPlanner':
            info = f'action={step.get("action")}, reason={step.get("reason")}'
        elif node == 'Handler':
            info = f'action={step.get("action")}, status={step.get("status")}'
            if 'explain_time_s' in step:
                info += f', time={step.get("explain_time_s")}s'
        else:
            info = str({k:v for k,v in step.items() if k != 'node'})
        print(f'  [{i+1}] {node:<15} {info}')
else:
    print('pipeline_trace.log not found — run ask() first')

=== PIPELINE TRACE ===
Query     : tổng hợp kiến thức tin học lớp 12
Timestamp : 2026-04-07 23:46:13
Total time: 11.88s

Steps (6):
  [1] ContextAnalyzer {'enriched': False}
  [2] IntentRouter    intent=explain, topic=kiến thức tin học lớp 12, time=1.1s
  [3] SessionManager  {'session_id': '5fe0b7c0', 'topic': 'kiến thức tin học lớp 12', 'intent': 'explain', 'total_messages': 0, 'has_quiz_state': False, 'has_slide_state': False}
  [4] ActionPlanner   action=explain_concept, reason=General concept explanation
  [5] RAG             strategy=broad, chunks=30, time=0.0s
  [6] Handler         action=explain_concept, status=success, time=10.75s
